# FinSight — QLoRA Fine-Tuning on Colab
**Runtime: GPU (T4)**  
Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Install dependencies
!pip install -q trl peft bitsandbytes transformers accelerate datasets wandb

In [ ]:
import os

# 2. Set your secrets — paste your actual tokens here
os.environ['HF_TOKEN'] = 'hf_...'       # your Hugging Face token
os.environ['WANDB_API_KEY'] = '...'     # optional, skip if you don't want W&B logging

HF_USERNAME = 'your-hf-username'        # e.g. 'sakshi-rathi'
HF_REPO = f'{HF_USERNAME}/finsight-llama-3.1-8b'

In [ ]:
# 3. Upload train.jsonl
# Run this cell, then click 'Choose Files' and upload your train.jsonl
from google.colab import files
uploaded = files.upload()  # upload phase1_finetune/data/train.jsonl

In [ ]:
# 4. Load dataset
from datasets import load_dataset

dataset = load_dataset('json', data_files='train.jsonl', split='train')
split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'Train: {len(train_ds)}, Eval: {len(eval_ds)}')

In [ ]:
# 5. Load model in 4-bit
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = 'meta-llama/Meta-Llama-3.1-8B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL, token=os.environ['HF_TOKEN'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    token=os.environ['HF_TOKEN'],
)
print('Model loaded!')

In [ ]:
# 6. Apply LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 7. Train
from trl import SFTConfig, SFTTrainer

def format_example(example):
    return f"{example['prompt']} {example['completion']}</s>"

report_to = 'wandb' if os.environ.get('WANDB_API_KEY') else 'none'

sft_config = SFTConfig(
    output_dir='./finsight-qlora',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    max_seq_length=1024,
    bf16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    report_to=report_to,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    formatting_func=format_example,
)

trainer.train()

In [ ]:
# 8. Push to Hugging Face Hub
trainer.save_model('./finsight-qlora')
tokenizer.save_pretrained('./finsight-qlora')

model.push_to_hub(HF_REPO, token=os.environ['HF_TOKEN'])
tokenizer.push_to_hub(HF_REPO, token=os.environ['HF_TOKEN'])
print(f'Done! Model at: https://huggingface.co/{HF_REPO}')

In [ ]:
# Alternative: Download adapters as zip (if you don't want to push to Hub)
import shutil
shutil.make_archive('finsight-adapters', 'zip', './finsight-qlora')
files.download('finsight-adapters.zip')